In [1]:
#!pip install transformers seqeval evaluate datasets

### Dataset 

In [2]:
from datasets import load_dataset

raw_datasets = load_dataset("lfcc/portuguese_ner")
print(raw_datasets)

/home/lfc/.myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3716
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 930
    })
})


In [9]:
label_list = raw_datasets["train"].features["ner_tags"].feature.names
print(label_list)

['O', 'B-Data', 'I-Data', 'B-Local', 'I-Local', 'B-Organizacao', 'I-Organizacao', 'B-Pessoa', 'I-Pessoa', 'B-Profissao', 'I-Profissao']


In [ ]:
raw_datasets["train"]["tokens"]

Column([['Filiação', ':', 'Antonio', 'Joaquim', 'Aguiar', 'e', 'Engracia', 'Maria', '.', 'Natural', 'e/ou', 'residente', 'em', 'CUNHA', ',', 'Santa', 'Maria', ',', 'actual', 'concelho', 'de', 'PAREDES', 'COURA', 'e', 'distrito', '(', 'ou', 'país', ')', 'Viana', 'do', 'Castelo', '.'], ['Filiação', ':', 'Domingos', 'Pires', 'e', 'Comba', 'Fernandes', '.', 'Natural', 'e/ou', 'residente', 'em', 'VALONGO', 'MILHAIS', ',', 'Sao', 'Goncalo', ',', 'actual', 'concelho', 'de', 'MURCA', 'e', 'distrito', '(', 'ou', 'país', ')', 'VILA', 'REAL', '.'], ['Termo', 'de', 'justificação', 'do', 'baptismo', 'de', 'Pedro', 'Gonçalves', 'Coques', ',', 'nascido', 'em', '29.06.1876', 'e', 'baptizado', '"', '(', '…', ')', 'por', 'dias', 'do', 'mês', 'de', 'Julho', 'do', 'dito', 'ano', ',', '(', '…', ')', '"', ',', 'na', 'igreja', 'do', 'Jardim', 'do', 'Mar', ',', 'Calheta', '.'], ['Doc.danificado', '.'], ['1898-11-01', '/', '1898-11-01']])

### Pre-Processing

In [3]:

checkpoint = "neuralmind/bert-base-portuguese-cased"
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [4]:
tokenized_inputs = tokenizer("A aula de SPLN é muito interessante!")
print(tokenized_inputs)

#print(tokenizer.convert_ids_to_tokens(tokenized_inputs["input_ids"]))

{'input_ids': [101, 177, 14190, 125, 9791, 22327, 22320, 253, 785, 11236, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
tokens = ["A", "aula", "de", "SPLN", "é", "muito", "interessante"]
tokenized_inputs = tokenizer(tokens, is_split_into_words=True)

print(tokenized_inputs)

{'input_ids': [101, 177, 14190, 125, 9791, 22327, 22320, 253, 785, 11236, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [17]:
print(len(tokens), len(tokenized_inputs["input_ids"]))

7 11


In [20]:
print(tokenizer.convert_ids_to_tokens(tokenized_inputs["input_ids"]))
tokens = ["A", "aula", "de", "SPLN", "é", "muito", "interessante"]

['[CLS]', 'A', 'aula', 'de', 'SP', '##L', '##N', 'é', 'muito', 'interessante', '[SEP]']


In [18]:
print(tokenized_inputs.word_ids())

[None, 0, 1, 2, 3, 3, 3, 4, 5, 6, None]


In [5]:
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    previous_id = None
    for word_id in word_ids:
        if word_id == None:
            new_labels.append(-100)
        elif previous_id != word_id:
            new_labels.append(labels[word_id])
        else:
            new_labels.append(-100)
        previous_id = word_id

    return new_labels

tokens = ["A", "aula", "de", "SPLN", "é", "muito", "interessante"]
tokenized_inputs = tokenizer(tokens, is_split_into_words=True)
labels = [0,0,0,1,0,0,0]

print(align_labels_with_tokens(labels,tokenized_inputs.word_ids()))

[-100, 0, 0, 0, 1, -100, -100, 0, 0, 0, -100]


In [6]:
def tokenize_dataset(dataset):
    new_dataset = []
    for data in dataset:
        tokenized_inputs = tokenizer(data["tokens"], is_split_into_words=True, truncation=True, max_length=512)
        new_labels = align_labels_with_tokens(data["ner_tags"], tokenized_inputs.word_ids())
        tokenized_inputs["labels"] = new_labels

        new_dataset.append(tokenized_inputs)
    return new_dataset

data_train = tokenize_dataset(raw_datasets["train"])
data_test = tokenize_dataset(raw_datasets["test"])

In [7]:
from datasets import Dataset
train_dataset = Dataset.from_list(data_train)
test_dataset = Dataset.from_list(data_test)
print(train_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 3716
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 930
})


In [10]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

label2id = {l : i for i, l in enumerate(label_list)}
id2label = {v: k for k, v in label2id.items()}
model = AutoModelForTokenClassification.from_pretrained(checkpoint, id2label=id2label , label2id = label2id )

Some weights of BertForTokenClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
batch_size = 8
output_model_name = "my_model"
args = TrainingArguments(
    output_model_name,
    report_to="none",
    eval_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size, #16
    per_device_eval_batch_size=batch_size, #16
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,      # Load the best model after training
    metric_for_best_model="f1",       # Or any metric you use for evaluation
    greater_is_better=True,           # True if higher F1/Accuracy is better
    #save_total_limit=1
    #push_to_hub=True,
)

In [12]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [13]:
import numpy as np
import evaluate
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Using the latest cached version of the module from /home/lfc/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--seqeval/541ae017dc683f85116597d48f621abc7b21b88dc42ec937c71af5415f0af63c (last modified on Wed May 22 18:44:28 2024) since it couldn't be found locally at evaluate-metric--seqeval, or remotely on the Hugging Face Hub.


In [14]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.065593,0.943069,0.964862,0.953841,0.983011
2,0.151700,0.061749,0.948995,0.971827,0.960275,0.984543


TrainOutput(global_step=930, training_loss=0.09948513892389113, metrics={'train_runtime': 109.5437, 'train_samples_per_second': 67.845, 'train_steps_per_second': 8.49, 'total_flos': 334427858237976.0, 'train_loss': 0.09948513892389113, 'epoch': 2.0})

In [20]:
from transformers import pipeline

classifier = pipeline("ner", model="./my_model/checkpoint-930", aggregation_strategy="first")

# Run inference
text = """O João Paulo, médico, vive no Porto (Portugal) desde 2024.
A Soraia Marques obeteve o seu mestrado na Universidade do Minho em 26 de Maio de 2021.
Depois de obter o grau, foi trabalhar para o o tribunal de Braga como juíza."
"""
classifier(text)

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[{'entity_group': 'Pessoa',
  'score': np.float32(0.98818445),
  'word': 'João Paulo',
  'start': 2,
  'end': 12},
 {'entity_group': 'Profissao',
  'score': np.float32(0.44979045),
  'word': 'médico',
  'start': 14,
  'end': 20},
 {'entity_group': 'Local',
  'score': np.float32(0.9788047),
  'word': 'Porto',
  'start': 30,
  'end': 35},
 {'entity_group': 'Local',
  'score': np.float32(0.978538),
  'word': 'Portugal',
  'start': 37,
  'end': 45},
 {'entity_group': 'Data',
  'score': np.float32(0.9688144),
  'word': '2024',
  'start': 53,
  'end': 57},
 {'entity_group': 'Pessoa',
  'score': np.float32(0.9923552),
  'word': 'Soraia Marques',
  'start': 61,
  'end': 75},
 {'entity_group': 'Organizacao',
  'score': np.float32(0.93042517),
  'word': 'Universidade do Minho',
  'start': 102,
  'end': 123},
 {'entity_group': 'Data',
  'score': np.float32(0.98656386),
  'word': '26 de Maio de 2021',
  'start': 127,
  'end': 145},
 {'entity_group': 'Organizacao',
  'score': np.float32(0.43681794)